# H0 – UT-HAR'ı SenseFi'den Drive'a kopyala (yalnızca Fatih, bir kez)
Bu notebook, SenseFi'nin herkese açık Drive klasöründeki **UT_HAR** verisini senin Drive'ındaki `ysa-wifi-csi/data/UT_HAR/` klasörüne kopyalar. Hücreleri sırayla çalıştır; ilk hücre Google hesabın için izin isteyecek.

In [ ]:
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build
srv = build('drive', 'v3')

SENSEFI_DATA = '1R0R8SlVbLI1iUFQCzh_mH90H_4CW2iwt'   # SenseFi 'Data' klasoru (herkese acik)
HEDEF_DATA   = '1g1A-DOlF1o4Iyv_YqfZZNjsR7483kU2r'   # MyDrive/ysa-wifi-csi/data
FOLDER = 'application/vnd.google-apps.folder'

def cocuklar(pid):
    q = f"'{pid}' in parents and trashed = false"
    sonuc, tok = [], None
    while True:
        r = srv.files().list(q=q, fields='nextPageToken, files(id, name, mimeType, size)',
                             pageToken=tok, supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        sonuc += r['files']; tok = r.get('nextPageToken')
        if not tok: return sonuc

for f in cocuklar(SENSEFI_DATA):
    print(f['name'], f['mimeType'], f.get('size'))

Yukarıda `UT_HAR` klasörü görünmeli. Şimdi onu ve alt klasörlerini kopyala:

In [ ]:
def klasor_ac(ad, ust):
    return srv.files().create(body={'name': ad, 'mimeType': FOLDER, 'parents': [ust]}, fields='id').execute()['id']

def kopyala(kaynak_id, hedef_ust, ad):
    yeni = klasor_ac(ad, hedef_ust)
    for f in cocuklar(kaynak_id):
        if f['mimeType'] == FOLDER:
            kopyala(f['id'], yeni, f['name'])
        else:
            srv.files().copy(fileId=f['id'], body={'name': f['name'], 'parents': [yeni]}, supportsAllDrives=True).execute()
            print('kopyalandi:', ad, '/', f['name'])
    return yeni

ut = [f for f in cocuklar(SENSEFI_DATA) if f['name'] == 'UT_HAR']
assert ut, 'UT_HAR klasoru bulunamadi; ustteki listeyi Claude ile paylas.'
kopyala(ut[0]['id'], HEDEF_DATA, 'UT_HAR')
print('\nTamam: MyDrive/ysa-wifi-csi/data/UT_HAR hazir.')

Bitti. Şimdi `H1_A_veri_kontrol.ipynb` notebook'unu çalıştırarak verinin doğru geldiğini kontrol et.